[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter8/multi-page-tables.ipynb)

# Chapter 8: Stitching Multi-Page Tables

When a table spans multiple pages in a PDF, most document parsers extract each page's fragment as a separate table, breaking the data's logical continuity. This creates two distinct problems:

1. **Redundancy** — if the header row is repeated on the next page, it can inject duplicate headers into the dataset.
2. **Ambiguity** — if the header is *not* repeated, the continuation becomes a headless matrix with no column context.

This notebook demonstrates the problem on a real PDF and implements a post-processing stitching layer that reassembles the fragments into a single coherent table.

**What you'll learn:**
- See how a document parser fragments a multi-page table into separate pieces
- Detect which fragments belong to the same logical table
- Standardize inconsistent column names across fragments
- Concatenate fragments into a single master DataFrame

**Prerequisites:** `pip install docling pandas`

## Setup

In [1]:
!pip install --quiet docling


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import requests
import pandas as pd

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

## Download and Parse the PDF

We use [Table 1.A — Population](https://unstats.un.org/unsd/demographic-social/products/worldswomen/annex_tables/Table1A.pdf) from the UN *World’s Women 2010* report. The entire 6-page PDF is a single table listing population data for every country, with headers repeated on each page.

In [3]:
url = "https://unstats.un.org/unsd/demographic-social/products/worldswomen/annex_tables/Table1A.pdf"
local_file = "Table1A.pdf"

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(local_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

pipeline_options = PdfPipelineOptions()
res = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
).convert(local_file)
doc = res.document

print(f"Tables extracted: {len(doc.tables)}")

Tables extracted: 6


## Inspecting the Fragments

Docling extracted **6 separate tables** from what is really one logical table. Each page produced its own fragment, and—critically—the column names differ from page to page because Docling re-interprets the repeated header independently each time.

In [4]:
# Collect fragments with page metadata
fragments = []
for table in doc.tables:
    df = table.export_to_dataframe(doc)
    page = table.prov[0].page_no if table.prov else 0
    fragments.append({'df': df, 'page': page})
    print(f"Page {page}: {df.shape[0]:>2} rows × {df.shape[1]} cols")

total_rows = sum(f['df'].shape[0] for f in fragments)
print(f"\nTotal: {total_rows} rows across {len(fragments)} fragments")

Page 1: 39 rows × 16 cols
Page 2: 39 rows × 16 cols
Page 3: 38 rows × 16 cols
Page 4: 38 rows × 16 cols
Page 5: 40 rows × 16 cols
Page 6: 11 rows × 16 cols

Total: 205 rows across 6 fragments


In [5]:
# Column names differ across pages despite being the same logical table
print("Fragment 0 columns (first 4):")
print(f"  {list(fragments[0]['df'].columns[:4])}\n")
print("Fragment 1 columns (first 4):")
print(f"  {list(fragments[1]['df'].columns[:4])}\n")

match = list(fragments[0]['df'].columns) == list(fragments[1]['df'].columns)
print(f"Column names match across fragments: {match}")
print(f"Column count matches: "
      f"{fragments[0]['df'].shape[1]} == {fragments[1]['df'].shape[1]}")

Fragment 0 columns (first 4):
  ['Country or area', 'Women', 'Women', 'Population (.1980 Women']

Fragment 1 columns (first 4):
  ['Country or area', 'Women', 'Women', 'Population (.1980 Women']

Column names match across fragments: False
Column count matches: 16 == 16


In [6]:
# First rows of fragment 0 (page 1)
fragments[0]['df'].head(4)

,Country or area,Women,Women,Population (.1980 Women,thousands.Men,).2010 Women,Men,Number ofmen per 100.women 2010,"Share population age 60.and above, 2010 ( % ) Women",Share population age 60.Men,of Total.1950- 1955,fertility.( birthsperwomen 1980- 1985,of Total.2010,rate ) Singulatemeanage at marriage 2005- Year,Women,Men
0,Africa,,,,,,,,,,,,,,,
1,Algeria,4 288,4 465,9 371,9 440,17 540,17 882,102,4,3,7.3,6.5,2.4,2002,29.5,33.0
2,Angola,2 113,2 034,3 992,3 862,9 631,9 362,97,2,2,7.0,7.2,5.8,..,..,..
3,Benin,1 095,955,1 837,1 723,4 560,4 651,102,3,2,5.7,7.0,5.5,2006,20.5,25.3


In [7]:
# First rows of fragment 1 (page 2) — note the "(continued)" marker row
fragments[1]['df'].head(4)

,Country or area,Women,Women,Population (.1980 Women,in thousands.Men,).2010 Women,Men,Number ofmen per 100.women 2010,"Share of population age 60.and above, 2010 ( % ) Women",Share of population age 60.Men,Total.1950- 1955,fertility.( birthsperwomen 1980- 1985 2005-,Total.2010,rate ) Singulatemeanage at marriage Year,Women,Men
0,Africa ( continued ),,,,,,,,,,,,,,,
1,Réunion,129,119,259,248,429,408,95,8,5,5.7,2.9,2.4,1999,30.5,32.8
2,Rwanda,1 093,1 069,2 705,2 492,5 296,4 981,94,2,2,7.8,8.3,5.4,2005,23.7,26.5
3,Sao Tome and Principe,27,33,48,47,83,82,98,3,3,6.2,6.2,3.9,1991,17.8,23.1


## The Stitching Algorithm

The stitching heuristic is straightforward:

1. **Consecutive pages** with **matching column count** → treat as fragments of the same table
2. **Standardize column names** from the first fragment (since the parser re-interprets headers on each page)
3. **Detect and remove duplicate header rows** if the first data row matches the column names
4. **Concatenate** into a single DataFrame

In [8]:
def has_duplicate_header(df):
    """Check if the first data row duplicates the column names."""
    if len(df) == 0:
        return False
    first_row = [str(v).strip() for v in df.iloc[0]]
    columns = [str(c).strip() for c in df.columns]
    return first_row == columns


def stitch_tables(extracted_tables):
    """
    Stitch consecutive table fragments with matching column counts.

    Parameters
    ----------
    extracted_tables : list of dict
        Each dict has 'df' (DataFrame) and 'page' (int).

    Returns
    -------
    list of DataFrame
        Tables with multi-page fragments merged where appropriate.
    """
    if not extracted_tables:
        return []

    tables = sorted(extracted_tables, key=lambda t: t['page'])
    result = []
    current = tables[0]['df'].copy()
    current_page = tables[0]['page']

    for entry in tables[1:]:
        next_df = entry['df'].copy()
        next_page = entry['page']

        if next_page == current_page + 1 and next_df.shape[1] == current.shape[1]:
            # Check for duplicate header before standardizing column names
            if has_duplicate_header(next_df):
                print(f'  Page {next_page}: removing dup header row')
                next_df = next_df.iloc[1:].reset_index(drop=True)

            # Standardize column names from the first fragment
            next_df.columns = current.columns

            print(f'  Stitching page {current_page} + {next_page}'
                  f' \u2192 {len(current) + len(next_df)} rows')
            current = pd.concat([current, next_df], ignore_index=True)
        else:
            result.append(current)
            current = next_df

        current_page = next_page

    result.append(current)
    return result

## Stitching the Table

In [9]:
stitched = stitch_tables(fragments)
print(f'\nResult: {len(stitched)} table(s)')

table = stitched[0]
print(f'Shape: {table.shape[0]} rows \u00d7 {table.shape[1]} columns')

  Stitching page 1 + 2 → 78 rows
  Stitching page 2 + 3 → 116 rows
  Stitching page 3 + 4 → 154 rows
  Stitching page 4 + 5 → 194 rows
  Stitching page 5 + 6 → 205 rows

Result: 1 table(s)
Shape: 205 rows × 16 columns


In [10]:
# Assign readable column names (Docling's parsed names are noisy)
COLUMNS = [
    'Country or area',
    'Pop. 1950 Women (000s)', 'Pop. 1950 Men (000s)',
    'Pop. 1980 Women (000s)', 'Pop. 1980 Men (000s)',
    'Pop. 2010 Women (000s)', 'Pop. 2010 Men (000s)',
    'Men per 100 Women',
    'Share 60+ Women (%)', 'Share 60+ Men (%)',
    'TFR 1950-55', 'TFR 1980-85', 'TFR 2005-10',
    'Marriage Age Year', 'Marriage Age Women', 'Marriage Age Men'
]
table.columns = COLUMNS
table.head(10)

,Country or area,Pop. 1950 Women (000s),Pop. 1950 Men (000s),Pop. 1980 Women (000s),Pop. 1980 Men (000s),Pop. 2010 Women (000s),Pop. 2010 Men (000s),Men per 100 Women,Share 60+ Women (%),Share 60+ Men (%),TFR 1950-55,TFR 1980-85,TFR 2005-10,Marriage Age Year,Marriage Age Women,Marriage Age Men
0,Africa,,,,,,,,,,,,,,,
1,Algeria,4 288,4 465,9 371,9 440,17 540,17 882,102,4,3,7.3,6.5,2.4,2002,29.5,33.0
2,Angola,2 113,2 034,3 992,3 862,9 631,9 362,97,2,2,7.0,7.2,5.8,..,..,..
3,Benin,1 095,955,1 837,1 723,4 560,4 651,102,3,2,5.7,7.0,5.5,2006,20.5,25.3
4,Botswana,213,200,504,481,988,989,100,4,3,6.5,6.0,2.9,2001,26.5,30.9
5,Burkina Faso,1 940,2 141,3 467,3 395,8 149,8 138,100,2,1,6.1,7.1,5.9,2003,19.4,26.1
6,Burundi,1 280,1 176,2 143,1 987,4 340,4 179,96,3,2,6.8,6.8,4.7,2002,23.7,26.1
7,Cameroon,2 277,2 189,4 580,4 500,9 978,9 981,100,3,3,5.7,6.4,4.7,2004,20.2,..
8,Cape Verde,80,66,157,133,267,245,92,4,2,6.6,6.1,2.8,2000,24.6,28.8
9,Central African Republic,673,654,1 154,1 115,2 292,2 214,97,4,3,5.5,6.0,4.8,1995,19.4,24.4


In [11]:
table.tail(10)

,Country or area,Pop. 1950 Women (000s),Pop. 1950 Men (000s),Pop. 1980 Women (000s),Pop. 1980 Men (000s),Pop. 2010 Women (000s),Pop. 2010 Men (000s),Men per 100 Women,Share 60+ Women (%),Share 60+ Men (%),TFR 1950-55,TFR 1980-85,TFR 2005-10,Marriage Age Year,Marriage Age Women,Marriage Age Men
195,Serbia,3 462,3 270,4 513,4 434,4 979,4 877,98,13,10,3.2,2.3,1.6,2002,25.9,29.8
196,Slovakia,1 782,1 681,2 531,2 446,2 787,2 625,94,13,8,3.5,2.3,1.3,2006,27.6,30.1
197,Slovenia,769,704,947,885,1 036,989,96,16,10,2.8,1.9,1.4,2006,31.2,33.4
198,Spain,14 483,13 526,19 121,18 406,22 956,22 360,97,16,12,2.6,1.9,1.4,2001,29.3,31.6
199,Sweden,3 521,3 493,4 193,4 118,4 679,4 614,99,17,13,2.2,1.6,1.9,2006,32.2,34.3
200,Switzerland,2 431,2 261,3 245,3 074,3 887,3 707,95,16,12,2.3,1.5,1.5,2007,29.4,32.2
201,The former Yugoslav Republic of Macedonia,613,616,886,909,1 023,1 020,100,11,8,5.3,2.3,1.4,1994,22.9,26.7
202,Ukraine,21 289,16 009,27 179,22 865,24 489,20 944,86,16,8,2.8,2.0,1.3,2007,23.1,25.9
203,United Kingdom,26 041,24 575,28 912,27 402,31 512,30 388,96,16,12,2.2,1.8,1.8,2001,26.3,28.1
204,United States of America,78 983,78 830,117 017,112 452,160 847,156 794,97,13,9,3.4,1.8,2.1,2000,26.0,27.8
